<a href="https://colab.research.google.com/github/CarlosDuarteti/Transcri-o-de-Audio/blob/main/Criando_transcri%C3%A7%C3%A3o_de_%C3%A1udio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Instalando os pacotes para realizar as transcrições
!pip install -q SpeechRecognition

In [2]:
#Importando o audio do pc local para o Colab

from base64 import b64decode
from google.colab import output
import speech_recognition as sr

# O Colab roda no navegador, então o microfone é acessado via JavaScript
def gravar_audio(segundos=5):
    codigo_js = """
    (async () => {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      //Criar uma div para exibir o botão de gravação
      const status = document.createElement('div');

        status.style.fontsize = "28x";
        status.style.fontWeight = "bold";
        status.style.margin = "15px";

        document.body.appendChild(status);
        status.innerHTML = "Fale algo! O programa irá gravar por 5 segundos.";

      await new Promise(r => setTimeout(r, SEGUNDOS * 1000));
      recorder.stop();
      status.remove();
      await new Promise(r => recorder.onstop = r);
      const blob = new Blob(chunks);
      const reader = new FileReader();
      const dataUrl = await new Promise(r => {
        reader.onload = () => r(reader.result);
        reader.readAsDataURL(blob);
      });
      return dataUrl;
    })()
    """.replace("SEGUNDOS", str(segundos))

    data_url = output.eval_js(codigo_js)      # executa o JS no navegador
    return b64decode(data_url.split(",")[1])  # converte base64 em bytes

In [3]:
audio_bytes = gravar_audio(5)

with open("audio.webm", "wb") as f:
  f.write(audio_bytes)

print("Áudio gravado com sucesso!")

MessageError: NotFoundError: Requested device not found

In [ ]:
#Realiza a conversão do audio para um formato que a web entende
!ffmpeg -y -i audio.webm audio.wav -loglevel quiet
print("Conversão concluida !")

Conversão concluida !


In [ ]:
#Realizando a transcrição do áudio para texto
recognize = sr.Recognizer()

with sr.AudioFile("audio.wav") as fonte:
    audio = recognize.record(fonte)

try:
    texto = recognize.recognize_google(audio, language="pt-BR")
    print("Transcrição ", texto)
except sr.UnknownValueError:
    print("Não entendi o áudio. Fale devagar ou mais perto do microfone.")
except sr.RequestError as e:
    print("Erro ao acessar a API do Google", e)

Transcrição  tem três aulas por semana um abraço


In [19]:
#CRIANDO UM LOOP PARA FALAR VARIAS VEZES

print("=== Transcrição contínua ===")
print("Fale algo. O programa grava 5 segundos e transcreve")
print("Para encerrar, digite: sair\n")

while True:
    print("Gravando aúdio por 5 segundos...")
    audio_bytes = gravar_audio(5)

    with open("audio.webm", "wb") as f:
        f.write(audio_bytes)

    !ffmpeg -y -i audio.webm audio.wav -loglevel quiet
    print("Conversão concluida !")

    recognize = sr.Recognizer()

    with sr.AudioFile("audio.wav") as fonte:
        audio = recognize.record(fonte)

    try:
        texto = recognize.recognize_google(audio, language="pt-BR")
        print("Transcrição ", texto)
    except sr.UnknownValueError:
        print("Não entendi o áudio. Fale devagar ou mais perto do microfone.")
    except sr.RequestError as e:
        print("Erro ao acessar a API do Google", e)

    comando = input("Digite ENTER para gravar novamente ou 'sair' para encerrar: ").strip().lower()
    if comando.lower() == "sair":
        break

print("Programa encerrado")

=== Transcrição contínua ===
Fale algo. O programa grava 5 segundos e transcreve
Para encerrar, digite: sair

Gravando aúdio por 5 segundos...


MessageError: NotFoundError: Requested device not found

In [1]:
from google.colab import files

arquivo_enviado = files.upload()
nome_arquivo = list(arquivo_enviado.keys())[0]
print("Nome do arquivo enviado: ", nome_arquivo)

Saving 1.wav to 1.wav
Nome do arquivo enviado:  1.wav


In [5]:
recognize = sr.Recognizer()

with sr.AudioFile(nome_arquivo) as fonte:
    audio = recognize.record(fonte)

    try:
        texto = recognize.recognize_google(audio, language="pt-BR")
        print("Transcrição ", texto)
    except sr.UnknownValueError:
        print("Não entendi o áudio. Fale devagar ou mais perto do microfone.")
    except sr.RequestError as e:
        print("Erro ao acessar a API do Google", e)

Transcrição  I'll Talk for fears Seconds to you can drag Nice my voice in the future


In [26]:
recognize = sr.Recognizer()

with sr.AudioFile(nome_arquivo) as fonte:
    audio = recognize.record(fonte)
idiomas = ["pt-BR", "en-US", "es-ES", "fr-FR"]

melhor_texto = ""
melhor_idioma = ""
melhor_confianca = 0.0

for idioma in idiomas:
    try:
        resultado = recognize.recognize_google(audio, language=idioma, show_all=True)
        if resultado:
            alternativa = resultado["alternative"][0]
            texto = alternativa["transcript"]
            confianca = alternativa.get("confidence", 0)
            print(f"{idioma}: Confiança: {confianca: .2f} -> {texto}")

            if confianca > melhor_confianca:
                melhor_texto = texto
                melhor_idioma = idioma

    except sr.UnknownValueError:
        print(f"{idioma}: Não entendi o aúdio.")
    except sr.RequestError as e:
        print(f"{idioma}: Erro de conexão", e)

print("\n*****   Melhor Resultado   *****")
print("Idioma Detectado: ", melhor_idioma)
print("Texto: ", melhor_texto)

try:
    texto = recognize.recognize_google(audio, language=melhor_idioma)
    print("Transcrição ", texto)
except sr.UnknownValueError:
    print("Não entendi o áudio. Fale devagar ou mais perto do microfone.")
except sr.RequestError as e:
    print("Erro ao acessar a API do Google", e)


pt-BR: Confiança:  0.82 -> I'll Talk for fears Seconds to you can drag Nice my voice in the future
en-US: Confiança:  0.99 -> I'll talk for a few seconds so you can recognize my voice in the future
es-ES: Confiança:  0.95 -> aerox for future nice my boys in the future
fr-FR: Confiança:  0.84 -> 50 sur YouTube

*****   Melhor Resultado   *****
Idioma Detectado:  fr-FR
Texto:  50 sur YouTube
Transcrição  50 sur YouTube
